# Optimal Mineral Extraction: Value Function Iteration

A pedagogical notebook for solving the deterministic Hotelling-style extraction problem with convex costs. We work through, in order:

1. **Analytical attempt** — undetermined coefficients with an LQ guess (and why it degenerates here).
2. **Method 1.** Discrete VFI on a joint $(Q, I)$ grid.
3. **Method 2.** Continuous-control VFI with interpolation.
4. **Method 4.** Endogenous Grid Method (EGM).
5. **Method 3.** Howard's policy iteration.

## The problem

Choose extraction $I_t \geq 0$ to maximize
$$
\sum_{t=0}^\infty \beta^t \big[\,p\, I_t - C(I_t)\,\big]
$$
subject to $Q_{t+1} = Q_t - I_t$ and $I_t \leq Q_t$, with $Q_0$ given. The Bellman equation is
$$
V(Q) \;=\; \max_{I \in [0, Q]} \;\big\{\, p\,I - C(I) \;+\; \beta\,V(Q - I)\,\big\}, \qquad V(0) = 0.
$$
We use a generic $C$ where useful and specialize to $C(I) = \tfrac{c}{2}\,I^2$. Calibration: $p = 1$, $c = 1$, $\beta = 0.95$, $Q_0 = 100$.


In [5]:
# One-time install (uncomment if needed)
# using Pkg; Pkg.add(["Plots", "Optim", "QuantEcon"])

using Plots, Optim, QuantEcon, LinearAlgebra, Printf
gr()

# Calibration
const β  = 0.95
const p  = 1.0
const c  = 1.0
const Q₀ = 100.0

# Functional forms (quadratic specialization of the generic C)
C(I)      = 0.5 * c * I^2          # cost
Cprime(I) = c * I                  # marginal cost
u(I)      = p*I - C(I)             # per-period payoff
Cprime_inv(x) = x / c              # inverse marginal cost (used in EGM)

# We use QuantEcon.LinInterp for 1-D linear interpolation:
#   itp = LinInterp(xgrid, ygrid)   then   itp(x)  evaluates at x
# It accepts any sorted xgrid (uniform or non-uniform — useful for EGM).
# Outside the grid it returns the boundary value (flat extrapolation).

println("Setup complete. β = $β, p = $p, c = $c, Q₀ = $Q₀")


LoadError: ArgumentError: Package Optim not found in current path:
- Run `import Pkg; Pkg.add("Optim")` to install the Optim package.


## 1. Analytical Attempt: Undetermined Coefficients

Following last week's class, we guess
$$
V(Q) \;=\; A + B\,Q + \tfrac{D}{2}\,Q^2.
$$
The interior FOC is $p - cI - \beta V'(Q-I) = 0$, which gives the candidate linear policy
$$
I^*(Q) \;=\; a + b\,Q, \qquad
a = \frac{p - \beta B}{c - \beta D}, \qquad
b = -\frac{\beta D}{c - \beta D}.
$$
Substituting back into the Bellman and matching coefficients on $Q^2$, $Q^1$, $Q^0$ gives a system for $(A,B,D)$. Doing the algebra:

**Match $Q^2$:** $D\big[1 - \beta(1-b)^2\big] = -c\,b^2$.  
**Combined with the FOC relation $D = bc/[\beta(b-1)]$:** the two equations reduce (after cancelation) to
$$
\beta(1-b) = 1 \quad \text{or} \quad b = 0.
$$

So the system has **two roots**:

| Root | $b$ | $D$ | Implied $V$ | Status |
|---|---|---|---|---|
| 1 | $0$ | $0$ | constant | degenerate |
| 2 | $1 - 1/\beta < 0$ | $c(1-\beta)/\beta > 0$ | strictly **convex** in $Q$ | spurious |

**Why does undetermined coefficients fail here?** Because the cost depends only on $I$, not on $Q$, the *only* way $Q$ enters the problem is through the constraint $I \leq Q$. That constraint induces a **kink** in $V$ at the threshold $\bar Q$ where it begins to bind, so the true $V$ is genuinely piecewise — not globally polynomial. No quadratic guess can match.

**Lagrangian view (why we still know what's happening).** Treat the resource as a single intertemporal budget, $\sum_t I_t = Q_0$. The FOC $\beta^t(p - c I_t) = \lambda$ gives
$$
I_t \;=\; \frac{p}{c} \;-\; \frac{\lambda}{c}\,\beta^{-t},
$$
so extraction declines geometrically — the discrete-time **Hotelling rule** (the *net* marginal benefit $p - cI$ rises at rate $1/\beta$). The horizon $T$ at which $I_T = 0$ is determined implicitly by $Q_0$, and that transcendental relationship is what blocks a closed-form $V(Q)$.

**Conclusion.** Even this stripped-down problem requires numerics. We turn to four methods.


In [ ]:
# Quick sanity check on the two roots
println("Root 1 (degenerate constant-extraction):")
println("  b = 0,  D = 0,  I*(Q) = p/c = $(p/c)  for all Q")
println("  ⇒ V(Q) is constant ⇒ economically wrong (V'(Q) > 0)")
println()

b₂ = 1 - 1/β
D₂ = c*(1-β)/β
@printf("Root 2 (spurious — convex V):\n")
@printf("  b = 1 - 1/β = %+0.4f   (negative — extraction would DECREASE in Q)\n", b₂)
@printf("  D = c(1-β)/β = %+0.4f   (positive — V would be CONVEX, violating concavity)\n", D₂)


## 2. Method 1 — Discrete VFI on a Joint Grid

Discretize $Q$ on a uniform grid and **restrict $I$ to grid increments** so that $Q' = Q - I$ stays on the grid. No interpolation is needed. The Bellman operator becomes
$$
(T V)(Q_i) \;=\; \max_{j \,\leq\, i}\;\big\{\, p\,(Q_i - Q_j) - C(Q_i - Q_j) \;+\; \beta\,V(Q_j)\,\big\}.
$$
We iterate $V \leftarrow T V$ until $\|TV - V\|_\infty < \text{tol}$. This is the most pedestrian implementation — its main pedagogical virtue is exposing the contraction mapping in raw form.


In [ ]:
function discrete_vfi(; N=501, Qmax=120.0, β=β, p=p, c=c,
                       tol=1e-8, maxiter=10_000)
    Qgrid = collect(range(0.0, Qmax; length=N))
    V     = zeros(N)
    Vnew  = similar(V)
    polidx = zeros(Int, N)
    history = Float64[]

    # Pre-compute payoff matrix: payoff[i,j] = u(Q[i]-Q[j]) for j ≤ i
    payoff = fill(-Inf, N, N)
    @inbounds for i in 1:N, j in 1:i
        ext = Qgrid[i] - Qgrid[j]
        payoff[i,j] = p*ext - 0.5*c*ext^2
    end

    iters = maxiter
    @inbounds for it in 1:maxiter
        for i in 1:N
            best, bestj = -Inf, 1
            for j in 1:i
                v = payoff[i,j] + β*V[j]
                if v > best
                    best, bestj = v, j
                end
            end
            Vnew[i]   = best
            polidx[i] = bestj
        end
        diff = maximum(abs.(Vnew .- V))
        push!(history, diff)
        copyto!(V, Vnew)
        if diff < tol
            iters = it
            break
        end
    end

    Ipol = [Qgrid[i] - Qgrid[polidx[i]] for i in 1:N]
    return (Qgrid=Qgrid, V=V, Ipol=Ipol, history=history, iters=iters)
end

@time res1 = discrete_vfi(N=501, Qmax=120.0, tol=1e-8)
println("Discrete VFI converged in $(res1.iters) iterations")


In [ ]:
p1a = plot(res1.Qgrid, res1.V, xlabel="Q", ylabel="V(Q)",
           title="Value (Discrete VFI)", lw=2, legend=false)
p1b = plot(res1.Qgrid, res1.Ipol, xlabel="Q", ylabel="I*(Q)",
           title="Policy (Discrete VFI)", lw=2, legend=false)
plot(p1a, p1b, layout=(1,2), size=(950, 360))


## 3. Method 2 — Continuous-Control VFI

Same grid for $Q$, but optimize $I$ over a **continuum** at each $Q_i$, using a 1-D optimizer. We interpolate $V$ between grid points so that the continuation value $V(Q_i - I)$ is well-defined for any $I \in [0, Q_i]$:
$$
V^{k+1}(Q_i) \;=\; \max_{I \in [0,\,Q_i]}\,\Big\{\,p\,I - C(I) + \beta\,\tilde V^{\,k}(Q_i - I)\,\Big\}.
$$
For modest grid sizes this typically yields much smoother and more accurate policies than Method 1 — at the cost of one inner optimization per grid point per iteration.


In [ ]:
function continuous_vfi(; N=201, Qmax=120.0, β=β, p=p, c=c,
                          tol=1e-8, maxiter=10_000)
    Qgrid = collect(range(0.0, Qmax; length=N))
    V     = zeros(N)
    Vnew  = similar(V)
    Ipol  = zeros(N)
    history = Float64[]

    iters = maxiter
    for it in 1:maxiter
        Vinterp = LinInterp(Qgrid, V)

        for i in 1:N
            Qi = Qgrid[i]
            if Qi == 0
                Vnew[i] = 0.0
                Ipol[i] = 0.0
                continue
            end
            obj(I) = -(p*I - 0.5*c*I^2 + β*Vinterp(Qi - I))   # negate ⇒ minimize
            r = optimize(obj, 0.0, Qi, GoldenSection(); abs_tol=1e-10)
            Vnew[i] = -Optim.minimum(r)
            Ipol[i] =  Optim.minimizer(r)
        end

        diff = maximum(abs.(Vnew .- V))
        push!(history, diff)
        copyto!(V, Vnew)
        if diff < tol
            iters = it
            break
        end
    end

    return (Qgrid=Qgrid, V=V, Ipol=Ipol, history=history, iters=iters)
end

@time res2 = continuous_vfi(N=201, Qmax=120.0, tol=1e-8)
println("Continuous-control VFI converged in $(res2.iters) iterations")


In [ ]:
p2a = plot(res2.Qgrid, res2.V, xlabel="Q", ylabel="V(Q)",
           title="Value (Continuous VFI)", lw=2, legend=false)
p2b = plot(res2.Qgrid, res2.Ipol, xlabel="Q", ylabel="I*(Q)",
           title="Policy (Continuous VFI)", lw=2, legend=false)
plot(p2a, p2b, layout=(1,2), size=(950, 360))


## 4. Method 4 — Endogenous Grid Method (EGM)

Track the **derivative** $V'(Q)$ instead of $V$ itself. The interior FOC and the envelope condition give
$$
\underbrace{p - C'(I)}_{\text{net MB today}} \;=\; \beta\,V'(Q'), \qquad
V'(Q) \;=\; \beta\,V'(Q - I).
$$
With quadratic cost $C'(I) = cI$, the FOC inverts in closed form:
$$
I \;=\; \frac{p - \beta\,V'(Q')}{c}.
$$

**Algorithm.** Pick a grid for *tomorrow's* stock $Q'$. At iteration $k$:
1. For each $Q'_j$, compute $I_j = \big(p - \beta V'_k(Q'_j)\big)/c$ (clipped at zero).
2. Recover *today's* stock: $Q_j = Q'_j + I_j$ — the **endogenous grid**.
3. Update marginal value via envelope: $V'_{k+1}(Q_j) = p - c\,I_j$.
4. Re-interpolate to the original grid for the next iteration.

For $Q$ below the smallest endogenous point, the constraint $I \leq Q$ binds and $V'(Q) = p - cQ$.

EGM **avoids the inner maximization entirely**, and for this problem the FOC inversion is trivial — so each iteration is cheap.


In [ ]:
function egm(; N=201, Qmax=120.0, β=β, p=p, c=c,
                tol=1e-9, maxiter=10_000)
    Qgrid = collect(range(0.0, Qmax; length=N))
    Vp     = fill(p, N)              # initial guess for V'(Q): start at p
    Vp_new = similar(Vp)
    history = Float64[]

    iters = maxiter
    for it in 1:maxiter
        # Step 1–2: endogenous grid
        Qend     = similar(Qgrid)
        Vp_endog = similar(Qgrid)
        for j in 1:N
            Qp   = Qgrid[j]
            Itry = (p - β*Vp[j]) / c
            Iv   = max(Itry, 0.0)
            Qend[j]     = Qp + Iv
            Vp_endog[j] = p - c*Iv          # = β V'_k(Q'_j)  by envelope (interior)
        end

        # Step 4: interpolate back to standard Qgrid (handle constrained region)
        perm = sortperm(Qend)
        Qend_s = Qend[perm]
        Vp_s   = Vp_endog[perm]
        itp = LinInterp(Qend_s, Vp_s)

        for i in 1:N
            Qi = Qgrid[i]
            if Qi <= Qend_s[1]
                Vp_new[i] = p - c*Qi          # constraint binds: I = Q
            else
                Vp_new[i] = itp(Qi)
            end
        end

        diff = maximum(abs.(Vp_new .- Vp))
        push!(history, diff)
        copyto!(Vp, Vp_new)
        if diff < tol
            iters = it
            break
        end
    end

    # Recover policy: I*(Q) = (p - V'(Q))/c, clipped to [0, Q]
    Ipol = clamp.((p .- Vp) ./ c, 0.0, Qgrid)

    # Recover V by integrating V': V(Q) = ∫₀^Q V'(q) dq, V(0) = 0
    V = zeros(N)
    for i in 2:N
        V[i] = V[i-1] + 0.5*(Vp[i] + Vp[i-1])*(Qgrid[i] - Qgrid[i-1])
    end

    return (Qgrid=Qgrid, V=V, Vp=Vp, Ipol=Ipol, history=history, iters=iters)
end

@time res4 = egm(N=201, Qmax=120.0, tol=1e-9)
println("EGM converged in $(res4.iters) iterations")


In [ ]:
p4a = plot(res4.Qgrid, res4.V, xlabel="Q", ylabel="V(Q)",
           title="Value (EGM)", lw=2, legend=false)
p4b = plot(res4.Qgrid, res4.Vp, xlabel="Q", ylabel="V'(Q)",
           title="Marginal Value (EGM)", lw=2, legend=false)
p4c = plot(res4.Qgrid, res4.Ipol, xlabel="Q", ylabel="I*(Q)",
           title="Policy (EGM)", lw=2, legend=false)
plot(p4a, p4b, p4c, layout=(1,3), size=(1200, 350))


## 5. Method 3 — Howard's Policy Iteration

For a fixed policy $g$, the value function $V^g$ satisfies a **linear** equation:
$$
V^g(Q_i) \;=\; u\big(g(Q_i)\big) \;+\; \beta\,V^g\!\big(Q_i - g(Q_i)\big).
$$
On the discrete grid (with $g$ mapping grid points to grid points, exactly as in Method 1) this becomes
$$
V^g \;=\; u^g \;+\; \beta\,P^g\,V^g \quad\Longrightarrow\quad V^g \;=\; (I - \beta\,P^g)^{-1} u^g,
$$
where $P^g$ is the deterministic transition matrix (one nonzero per row).

**Algorithm.**
1. Start from a guess $g_0$ (e.g., $g_0 \equiv 0$).
2. **Policy evaluation:** solve the linear system above for $V^{g_k}$.
3. **Policy improvement:** $g_{k+1}(Q_i) = \arg\max_j \{u(Q_i - Q_j) + \beta V^{g_k}(Q_j)\}$.
4. Stop when $g_{k+1} = g_k$.

Howard converges in **far fewer outer iterations** than VFI — typically $O(\log N)$ — because each policy evaluation captures the full geometric series implicit in the fixed-policy value. The cost per outer iteration is one linear solve.


In [ ]:
function howard(; N=501, Qmax=120.0, β=β, p=p, c=c, maxiter=200)
    Qgrid = collect(range(0.0, Qmax; length=N))

    payoff = fill(-Inf, N, N)
    @inbounds for i in 1:N, j in 1:i
        ext = Qgrid[i] - Qgrid[j]
        payoff[i,j] = p*ext - 0.5*c*ext^2
    end

    polidx = collect(1:N)             # initial policy: g(Q[i]) = 0  (j = i, no extraction)
    history = Int[]
    Iden = Matrix{Float64}(LinearAlgebra.I, N, N)
    Vg   = zeros(N)

    iters = maxiter
    for it in 1:maxiter
        # Policy evaluation: build P^g and u^g, solve (I - βP^g) V^g = u^g
        Pg = zeros(N, N)
        ug = zeros(N)
        for i in 1:N
            j = polidx[i]
            Pg[i, j] = 1.0
            ug[i]    = payoff[i, j]
        end
        Vg = (Iden - β*Pg) \ ug

        # Policy improvement (greedy step)
        polnew = similar(polidx)
        for i in 1:N
            best, bestj = -Inf, 1
            for j in 1:i
                v = payoff[i,j] + β*Vg[j]
                if v > best
                    best, bestj = v, j
                end
            end
            polnew[i] = bestj
        end
        push!(history, count(polnew .!= polidx))
        if polnew == polidx
            iters = it
            break
        end
        polidx = polnew
    end

    Ipol = [Qgrid[i] - Qgrid[polidx[i]] for i in 1:N]
    return (Qgrid=Qgrid, V=Vg, polidx=polidx, Ipol=Ipol,
            history=history, iters=iters)
end

@time res3 = howard(N=501, Qmax=120.0)
println("Howard's PI converged in $(res3.iters) outer iterations")
println("Policy changes per iteration: ", res3.history)


## 6. Comparing the Methods

All four methods solve the same problem and should agree (up to discretization error). The interesting differences are in **iteration count**, **per-iteration cost**, and **policy smoothness**.


In [ ]:
# Comparison plots
pV = plot(xlabel="Q", ylabel="V(Q)", title="Value Function",
          legend=:bottomright)
plot!(pV, res1.Qgrid, res1.V, label="Discrete VFI",       lw=2)
plot!(pV, res2.Qgrid, res2.V, label="Continuous VFI",     lw=2, ls=:dash)
plot!(pV, res4.Qgrid, res4.V, label="EGM",                lw=2, ls=:dot)
plot!(pV, res3.Qgrid, res3.V, label="Howard PI",          lw=2, ls=:dashdot)

pI = plot(xlabel="Q", ylabel="I*(Q)", title="Policy Function",
          legend=:bottomright)
plot!(pI, res1.Qgrid, res1.Ipol, label="Discrete VFI",    lw=2)
plot!(pI, res2.Qgrid, res2.Ipol, label="Continuous VFI",  lw=2, ls=:dash)
plot!(pI, res4.Qgrid, res4.Ipol, label="EGM",             lw=2, ls=:dot)
plot!(pI, res3.Qgrid, res3.Ipol, label="Howard PI",       lw=2, ls=:dashdot)

plot(pV, pI, layout=(1,2), size=(1050, 400))


In [ ]:
# Iteration counts
println("─"^48)
@printf("  Discrete VFI       : %5d  iterations\n", res1.iters)
@printf("  Continuous VFI     : %5d  iterations\n", res2.iters)
@printf("  EGM                : %5d  iterations\n", res4.iters)
@printf("  Howard PI          : %5d  outer steps\n", res3.iters)
println("─"^48)


## 7. Simulated Optimal Path

Starting from $Q_0 = 100$, simulate forward under each method's policy. The Hotelling intuition predicts a geometrically declining extraction profile — let's see it.


In [ ]:
function simulate_path(Qgrid, Ipol, Q0, T)
    Qpath = zeros(T+1)
    Ipath = zeros(T)
    Qpath[1] = Q0
    itp = LinInterp(Qgrid, Ipol)
    for t in 1:T
        Ipath[t]   = clamp(itp(Qpath[t]), 0.0, Qpath[t])
        Qpath[t+1] = Qpath[t] - Ipath[t]
    end
    return Qpath, Ipath
end

T = 250
Qp1, Ip1 = simulate_path(res1.Qgrid, res1.Ipol, Q₀, T)
Qp2, Ip2 = simulate_path(res2.Qgrid, res2.Ipol, Q₀, T)
Qp4, Ip4 = simulate_path(res4.Qgrid, res4.Ipol, Q₀, T)
Qp3, Ip3 = simulate_path(res3.Qgrid, res3.Ipol, Q₀, T)

pQ = plot(0:T, Qp1, lw=2, label="Discrete VFI", xlabel="t", ylabel="Q_t",
          title="Stock path", legend=:topright)
plot!(pQ, 0:T, Qp2, lw=2, ls=:dash,    label="Continuous VFI")
plot!(pQ, 0:T, Qp4, lw=2, ls=:dot,     label="EGM")
plot!(pQ, 0:T, Qp3, lw=2, ls=:dashdot, label="Howard")

pIp = plot(0:T-1, Ip1, lw=2, label="Discrete VFI", xlabel="t", ylabel="I_t",
           title="Extraction path", legend=:topright)
plot!(pIp, 0:T-1, Ip2, lw=2, ls=:dash,    label="Continuous VFI")
plot!(pIp, 0:T-1, Ip4, lw=2, ls=:dot,     label="EGM")
plot!(pIp, 0:T-1, Ip3, lw=2, ls=:dashdot, label="Howard")

plot(pQ, pIp, layout=(1,2), size=(1050, 400))


## 8. Convergence Speed

Sup-norm error per iteration on a log scale. The slope of plain VFI is set by the contraction modulus $\beta = 0.95$ — slow. EGM and the policy-improvement step in Howard achieve much faster effective convergence.


In [ ]:
ph = plot(xlabel="Iteration", ylabel="Sup-norm change", yscale=:log10,
          title="Convergence (log scale)", legend=:topright)
plot!(ph, 1:length(res1.history), res1.history, label="Discrete VFI",   lw=2)
plot!(ph, 1:length(res2.history), res2.history, label="Continuous VFI", lw=2, ls=:dash)
plot!(ph, 1:length(res4.history), res4.history, label="EGM (‖V'_{k+1}-V'_k‖)", lw=2, ls=:dot)
# Howard tracks # of policy changes, not value change — show separately:
plot(ph, size=(750, 420))


In [ ]:
ph_howard = plot(1:length(res3.history), res3.history,
                  xlabel="Outer iteration", ylabel="# of grid points where policy changed",
                  title="Howard's Policy Iteration",
                  lw=2, marker=:circle, legend=false)
plot(ph_howard, size=(700, 360))


## 9. Take-aways for Class

1. **Why undetermined coefficients fails.** With state-independent costs, the only state-dependence comes from $I \leq Q$. That kink prevents a globally polynomial $V$. Numerical methods are unavoidable.
2. **Hotelling visible in the simulation.** $I_t$ declines (approximately) geometrically — exactly the Lagrangian's $I_t = p/c - K\beta^{-t}$, ramping down toward zero as the resource depletes.
3. **Discrete vs continuous control.** Discrete VFI (Method 1) is conceptually transparent but produces a "step-shaped" policy on the grid. Continuous-control VFI (Method 2) recovers a smooth policy at the cost of one inner optimization per grid point per iteration.
4. **EGM speedup.** EGM dodges the inner maximization. For quadratic cost the FOC inverts in closed form, so each iteration is essentially free. EGM also generalizes cleanly to consumption-savings problems students will see next.
5. **Howard's win.** With $\beta = 0.95$, plain VFI converges at rate $\beta^k$ — about 450 iterations to reach $10^{-8}$ from a starting error of order $V_{\max}$. Howard typically finishes in 5–15 outer iterations because each policy evaluation captures the full geometric series implicit at fixed policy.

## Suggested Exercises

- Replace $C(I) = \tfrac{c}{2} I^2$ with the generic $C(I) = \tfrac{c}{1+\gamma}\,I^{1+\gamma}$ and re-run all four methods. EGM still works because $C'$ is invertible in closed form.
- Add stochastic prices $p_t = \rho\,p_{t-1} + \varepsilon_t$. The state becomes $(Q, p)$; build a Markov chain for $p$.
- Add a discovery process $Q_{t+1} = Q_t - I_t + \eta_t$ with $\eta_t \geq 0$ random. Now $V$ is no longer monotonic in time even at fixed $Q$.
- Verify numerically the discrete-time Hotelling rule: $\big(p - C'(I_t)\big) \cdot 1/\beta \approx p - C'(I_{t+1})$ along the simulated path.
